In [1]:
import json
from urllib.request import Request, urlopen

TEAM_ID = "TEAM_47"
API_KEY = " oc_AP7Mcb05OZ6vpa42KbEULwDzp1yMSIHI"
API_URL = " https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"

def api(payload):
    req = Request(API_URL, data=json.dumps(payload).encode(),

                    headers={"Content-Type":"application/json", "X-API-Key":API_KEY})

    with urlopen(req, timeout=30) as r:
        return json.load(r)
    
spec = api({"action":"spec", "team_id":TEAM_ID})
SPACE = spec["hyperparameters"]
print("Model:", spec["model"])
for name, values in SPACE.items():
    print(name, ":", values)

Model: RandomForestRegressor
bootstrap : [True, False]
max_depth : [3, 4, 5, 7, 10, None]
max_features : [0.4, 0.6, 0.8, 1]
n_estimators : [3, 5, 8, 12]
min_samples_leaf : [1, 2, 4, 8]
min_samples_split : [2, 5, 10, 20]


In [2]:
# ============================================================
# CELL 2: CREATE ALL POSSIBLE HYPERPARAMETER CONFIGURATIONS
# ============================================================
#
# IMPORTANT:
# This cell makes ZERO API calls.
#
# We are only creating all possible combinations locally.
# There are 2560 possible combinations, but we will NOT
# send all of them to the API.
# ============================================================

from itertools import product

# Get parameter names in the same order as SPACE
parameter_names = list(SPACE.keys())

# Get all possible values for each parameter
parameter_values = [SPACE[name] for name in parameter_names]

# Create all possible combinations locally
all_candidates = []

for combination in product(*parameter_values):

    # Convert the tuple of values into a dictionary
    params = dict(zip(parameter_names, combination))

    all_candidates.append(params)

# Display information
print("Total possible configurations:", len(all_candidates))

print("\nFirst 3 configurations:")
for params in all_candidates[:3]:
    print(params)

Total possible configurations: 3072

First 3 configurations:
{'bootstrap': True, 'max_depth': 3, 'max_features': 0.4, 'n_estimators': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
{'bootstrap': True, 'max_depth': 3, 'max_features': 0.4, 'n_estimators': 3, 'min_samples_leaf': 1, 'min_samples_split': 5}
{'bootstrap': True, 'max_depth': 3, 'max_features': 0.4, 'n_estimators': 3, 'min_samples_leaf': 1, 'min_samples_split': 10}


In [3]:
# ============================================================
# CELL 3: API CALL CACHE
# ============================================================
#
# The most important rule of this competition:
#
# NEVER make an API call for the exact same configuration twice.
#
# results_cache will remember every configuration we have
# evaluated and its corresponding loss.
# ============================================================

results_cache = {}

# Counter for actual API calls
api_call_count = 0

# Store the best configuration found so far
best_params = None
best_loss = float("inf")


def make_key(params):
    """
    Convert a parameter dictionary into a unique, hashable key.

    Sorting ensures that the key is consistent even if the
    dictionary order changes.
    """

    return tuple(sorted(params.items()))


print("Cache initialized.")
print("API calls made so far:", api_call_count)

Cache initialized.
API calls made so far: 0


In [4]:
# ============================================================
# CELL 4: EVALUATION FUNCTION
# ============================================================
#
# This function will eventually be the ONLY place where
# our optimization algorithm is allowed to call the API.
#
# For now, we are NOT running it.
# ============================================================

def evaluate(params):
    """
    Evaluate one hyperparameter configuration.

    Rules:
    1. Check whether this configuration was already evaluated.
    2. If yes -> return the stored loss WITHOUT an API call.
    3. If no -> make exactly ONE API call.
    4. Store the result.
    """

    global api_call_count
    global best_params
    global best_loss

    # Create unique key for this configuration
    key = make_key(params)

    # --------------------------------------------------------
    # STEP 1: CHECK CACHE
    # --------------------------------------------------------

    if key in results_cache:

        print("Already evaluated.")
        print("No API call made.")

        return results_cache[key]

    # --------------------------------------------------------
    # STEP 2:
    # ACTUAL API CALL WILL GO HERE
    # --------------------------------------------------------
    #
    # DO NOT RUN THIS FUNCTION YET.
    #
    # We still need the exact Oracle evaluation action.
    #
    # It will eventually look conceptually like:
    #
    # response = api({
    #     "action": "...",
    #     "team_id": TEAM_ID,
    #     "parameters": params
    # })
    #
    # loss = response["loss"]
    #
    # --------------------------------------------------------

    raise NotImplementedError(
        "Oracle evaluation action is not inserted yet."
    )

In [5]:
# ============================================================
# CELL 4: INSPECT THE EXACT SEARCH SPACE
# ============================================================
#
# This cell makes ZERO API calls.
#
# We want to verify exactly how many values are available
# for each hyperparameter before designing the optimizer.
# ============================================================

print("EXACT HYPERPARAMETER SEARCH SPACE")
print("=" * 60)

total = 1

for name, values in SPACE.items():

    print(f"{name}:")
    print(f"  Values: {values}")
    print(f"  Number of values: {len(values)}")

    # Multiply the number of choices to get the total
    # number of possible configurations.
    total *= len(values)

print("=" * 60)
print("TOTAL POSSIBLE CONFIGURATIONS:", total)

EXACT HYPERPARAMETER SEARCH SPACE
bootstrap:
  Values: [True, False]
  Number of values: 2
max_depth:
  Values: [3, 4, 5, 7, 10, None]
  Number of values: 6
max_features:
  Values: [0.4, 0.6, 0.8, 1]
  Number of values: 4
n_estimators:
  Values: [3, 5, 8, 12]
  Number of values: 4
min_samples_leaf:
  Values: [1, 2, 4, 8]
  Number of values: 4
min_samples_split:
  Values: [2, 5, 10, 20]
  Number of values: 4
TOTAL POSSIBLE CONFIGURATIONS: 3072


In [6]:
# ============================================================
# CELL 5: VERIFY CANDIDATE SPACE
# ============================================================
#
# Again, ZERO API calls.
#
# We check that every generated configuration is unique.
# ============================================================

candidate_keys = {
    make_key(params)
    for params in all_candidates
}

print("Number of candidates:", len(all_candidates))
print("Number of unique candidates:", len(candidate_keys))

if len(all_candidates) == len(candidate_keys):
    print("✓ All candidates are unique.")
else:
    print("⚠ Duplicate candidates found.")

Number of candidates: 3072
Number of unique candidates: 3072
✓ All candidates are unique.


In [7]:
# ============================================================
# CELL 6: CHECK WHAT API ACTIONS ARE AVAILABLE
# ============================================================

# We are NOT calling the API here.
# This only displays what we already know about the spec.

print("Model:", spec["model"])
print("\nHyperparameters:")
for name, values in SPACE.items():
    print(f"  {name}: {values}")

print("\nTotal configurations:", len(all_candidates))
print("API calls used so far:", api_call_count)

Model: RandomForestRegressor

Hyperparameters:
  bootstrap: [True, False]
  max_depth: [3, 4, 5, 7, 10, None]
  max_features: [0.4, 0.6, 0.8, 1]
  n_estimators: [3, 5, 8, 12]
  min_samples_leaf: [1, 2, 4, 8]
  min_samples_split: [2, 5, 10, 20]

Total configurations: 3072
API calls used so far: 0


In [8]:
# ============================================================
# CELL 6: SEE THE COMPLETE ORACLE SPECIFICATION
# ============================================================

print("COMPLETE SPECIFICATION:")
print("=" * 70)

print(spec)

print("=" * 70)


COMPLETE SPECIFICATION:
{'team_id': 'TEAM_47', 'oracle_id': 'ORACLE_12', 'model': 'RandomForestRegressor', 'case_id': 2, 'n_configurations': 3072, 'hyperparameters': {'bootstrap': [True, False], 'max_depth': [3, 4, 5, 7, 10, None], 'max_features': [0.4, 0.6, 0.8, 1], 'n_estimators': [3, 5, 8, 12], 'min_samples_leaf': [1, 2, 4, 8], 'min_samples_split': [2, 5, 10, 20]}}


In [9]:
# ============================================================
# CELL 7: FIRST HYPERPARAMETER EVALUATION
# ============================================================
#
# IMPORTANT:
# This is our FIRST evaluation API call.
#
# We are deliberately testing ONE configuration only.
# The returned loss will become our baseline.
# ============================================================

# Choose ONE baseline configuration from the allowed values.
baseline_params = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.8,
    "n_estimators": 8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

print("Baseline configuration:")
for name, value in baseline_params.items():
    print(f"  {name}: {value}")

# ------------------------------------------------------------
# MAKE THE EVALUATION REQUEST
# ------------------------------------------------------------

response = api({
    "action": "evaluate",
    "team_id": TEAM_ID,
    "hyperparameters": baseline_params
})

print("\nOracle response:")
print(response)

Baseline configuration:
  bootstrap: True
  max_depth: 7
  max_features: 0.8
  n_estimators: 8
  min_samples_leaf: 2
  min_samples_split: 5


HTTPError: HTTP Error 400: Bad Request

In [10]:
# ============================================================
# ORACLE QUERY FUNCTION
# ============================================================

def oracle_query(params):
    """
    Sends one hyperparameter configuration to the Oracle
    and returns the loss.
    
    The Oracle expects:
        action = "query"
        team_id = TEAM_ID
        params = our hyperparameter dictionary
    """
    
    result = api({
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    })
    
    # Extract the loss returned by the Oracle
    loss = float(result["loss"])
    
    return loss

In [11]:
# ============================================================
# TEST ONE CONFIGURATION
# ============================================================

test_params = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.8,
    "n_estimators": 8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

# Send this configuration to the Oracle
test_loss = oracle_query(test_params)

print("Test parameters:")
print(test_params)

print("\nLoss returned by Oracle:", test_loss)

URLError: <urlopen error [WinError 10054] An existing connection was forcibly closed by the remote host>

In [ ]:
import json
from urllib.request import Request, urlopen

TEAM_ID = "TEAM_47"
API_KEY = " oc_AP7Mcb05OZ6vpa42KbEULwDzp1yMSIHI"
API_URL = " https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"

def api(payload):
    req = Request(
        API_URL,
        data=json.dumps(payload).encode(),
        headers={
            "Content-Type": "application/json",
            "X-API-Key": API_KEY
        }
    )
    
    with urlopen(req, timeout=120) as r:
        return json.load(r)

spec = api({
    "action": "spec",
    "team_id": TEAM_ID
})

SPACE = spec["hyperparameters"]

print("Model:", spec["model"])

for name, values in SPACE.items():
    print(name, ":", values)

Model: RandomForestRegressor
bootstrap : [True, False]
max_depth : [3, 4, 5, 7, 10, None]
max_features : [0.4, 0.6, 0.8, 1]
n_estimators : [3, 5, 8, 12]
min_samples_leaf : [1, 2, 4, 8]
min_samples_split : [2, 5, 10, 20]


In [13]:
from itertools import product

param_names = list(SPACE.keys())

all_candidates = [
    dict(zip(param_names, values))
    for values in product(*SPACE.values())
]

print("Total candidates:", len(all_candidates))

Total candidates: 3072


In [14]:
# ============================================================
# ORACLE QUERY FUNCTION
# ============================================================

def oracle_query(params):
    """
    Sends one hyperparameter configuration to the Oracle
    and returns the loss.
    
    The Oracle expects:
        action = "query"
        team_id = TEAM_ID
        params = our hyperparameter dictionary
    """
    
    result = api({
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    })
    
    # Extract the loss returned by the Oracle
    loss = float(result["loss"])
    
    return loss

In [15]:
# ============================================================
# TEST ONE CONFIGURATION
# ============================================================

test_params = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.8,
    "n_estimators": 8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

# Send this configuration to the Oracle
test_loss = oracle_query(test_params)

print("Test parameters:")
print(test_params)

print("\nLoss returned by Oracle:", test_loss)

Test parameters:
{'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 8, 'min_samples_leaf': 2, 'min_samples_split': 5}

Loss returned by Oracle: 7.29334422222043


In [16]:
# ============================================================
# CACHE + API CALL COUNTER
# ============================================================

# Stores every configuration that we have already queried
# so that we never ask the Oracle for the same configuration twice.
results_cache = {}

# Counts only actual Oracle queries
api_call_count = 0

# Store our current best configuration
best_params = test_params.copy()

# Store the loss of our current best configuration
best_loss = test_loss

print("Initial best loss:", best_loss)
print("Initial best parameters:", best_params)
print("API calls used:", api_call_count)

Initial best loss: 7.29334422222043
Initial best parameters: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 8, 'min_samples_leaf': 2, 'min_samples_split': 5}
API calls used: 0


In [20]:
# ============================================================
# EVALUATE A CONFIGURATION
# ============================================================

def evaluate(params):
    """
    Evaluate one hyperparameter configuration.

    If we have already tested this exact configuration,
    use the cached loss instead of calling the Oracle again.
    """

    global api_call_count

    # Create a unique key for this configuration
    key = tuple(sorted(params.items()))

    # --------------------------------------------------------
    # CHECK WHETHER THIS CONFIGURATION WAS ALREADY TESTED
    # --------------------------------------------------------
    if key in results_cache:
        print("Already tested -> using cached result")
        return results_cache[key]

    # --------------------------------------------------------
    # NEW CONFIGURATION -> CALL ORACLE
    # --------------------------------------------------------

    api_call_count += 1

    print("New Oracle query. Call number:", api_call_count)

    loss = oracle_query(params)

    # Save the result
    results_cache[key] = loss

    return loss

In [21]:
# ============================================================
# OPTIMIZE ONE PARAMETER
# ============================================================

def optimize_parameter(current_params, parameter_name):
    """
    Optimize one hyperparameter while keeping
    all other hyperparameters fixed.

    Every allowed value of the selected parameter
    is tested.

    The value producing the lowest loss is retained.
    """

    global best_params
    global best_loss

    print("\n" + "=" * 60)
    print("Optimizing:", parameter_name)
    print("=" * 60)

    # Current value
    current_value = current_params[parameter_name]

    print("Current value:", current_value)
    print("Current best loss:", best_loss)

    # Keep track of the best value for this parameter
    parameter_best_value = current_value
    parameter_best_loss = best_loss

    # --------------------------------------------------------
    # TRY EVERY ALLOWED VALUE
    # --------------------------------------------------------

    for value in SPACE[parameter_name]:

        # Make a copy of the current configuration
        candidate = current_params.copy()

        # Change ONLY the parameter we are currently optimizing
        candidate[parameter_name] = value

        # Evaluate the candidate
        loss = evaluate(candidate)

        print(
            parameter_name,
            "=",
            value,
            "--> loss =",
            round(loss, 6)
        )

        # ----------------------------------------------------
        # CHECK WHETHER THIS IS BETTER
        # ----------------------------------------------------

        if loss < parameter_best_loss:

            parameter_best_loss = loss
            parameter_best_value = value

            print(">>> New best found!")

    # --------------------------------------------------------
    # KEEP THE BEST VALUE FOUND FOR THIS PARAMETER
    # --------------------------------------------------------

    current_params[parameter_name] = parameter_best_value

    best_params = current_params.copy()
    best_loss = parameter_best_loss

    print("\nBest", parameter_name, "=", parameter_best_value)
    print("Best loss =", best_loss)

    return current_params

In [19]:
# ============================================================
# FIRST OPTIMIZATION ROUND
# ============================================================

current_params = best_params.copy()

# We will optimize one parameter at a time.
parameter_order = [
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "min_samples_split",
    "max_features",
    "bootstrap"
]

for parameter in parameter_order:

    current_params = optimize_parameter(
        current_params,
        parameter
    )

print("\n" + "=" * 60)
print("FIRST ROUND COMPLETE")
print("=" * 60)

print("Best parameters:")
print(best_params)

print("\nBest loss:", best_loss)

print("\nOracle calls used:", api_call_count)


Optimizing: n_estimators
Current value: 8
Current loss: 7.29334422222043
n_estimators = 3 --> loss = 8.342227
n_estimators = 5 --> loss = 7.433057
n_estimators = 8 --> loss = 7.293344
n_estimators = 12 --> loss = 6.821800
  >>> New best found!

Best value for n_estimators : 12
Best loss: 6.82180027961698

Optimizing: max_depth
Current value: 7
Current loss: 6.82180027961698


TimeoutError: The read operation timed out

In [22]:
# ============================================================
# RESTORE CURRENT BEST CONFIGURATION
# ============================================================

best_params = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.8,
    "n_estimators": 12,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

best_loss = 6.82180027961698

current_params = best_params.copy()

print("Current best parameters:")
print(current_params)

print("\nCurrent best loss:")
print(best_loss)

print("\nOracle calls recorded:")
print(api_call_count)

Current best parameters:
{'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}

Current best loss:
6.82180027961698

Oracle calls recorded:
4


In [23]:
# ============================================================
# OPTIMIZE MAX_DEPTH
# ============================================================

current_params = optimize_parameter(
    current_params,
    "max_depth"
)

print("\n" + "=" * 60)
print("RESULT AFTER MAX_DEPTH")
print("=" * 60)

print("Best parameters:")
print(best_params)

print("\nBest loss:")
print(best_loss)

print("\nOracle calls attempted:")
print(api_call_count)


Optimizing: max_depth
Current value: 7
Current best loss: 6.82180027961698
New Oracle query. Call number: 5
max_depth = 3 --> loss = 10.347232
New Oracle query. Call number: 6
max_depth = 4 --> loss = 8.04029
New Oracle query. Call number: 7


TimeoutError: The read operation timed out

In [24]:
# ============================================================
# OPTIMIZE ONE PARAMETER - TIMEOUT SAFE
# ============================================================

from urllib.error import HTTPError
from socket import timeout as SocketTimeout


def optimize_parameter(current_params, parameter_name):
    """
    Optimize one hyperparameter at a time.

    If an Oracle request times out, that configuration is skipped
    and the optimization continues with the remaining values.
    """

    global best_params
    global best_loss

    print("\n" + "=" * 60)
    print("Optimizing:", parameter_name)
    print("=" * 60)

    current_value = current_params[parameter_name]

    print("Current value:", current_value)
    print("Current best loss:", best_loss)

    # Start with the current best
    parameter_best_value = current_value
    parameter_best_loss = best_loss

    # --------------------------------------------------------
    # TRY EVERY ALLOWED VALUE
    # --------------------------------------------------------

    for value in SPACE[parameter_name]:

        # Create candidate configuration
        candidate = current_params.copy()

        # Change only this parameter
        candidate[parameter_name] = value

        print("\nTrying", parameter_name, "=", value)

        try:

            # Evaluate candidate
            loss = evaluate(candidate)

            print(
                parameter_name,
                "=",
                value,
                "--> loss =",
                round(loss, 6)
            )

            # Check if this is better
            if loss < parameter_best_loss:

                parameter_best_loss = loss
                parameter_best_value = value

                print(">>> NEW BEST FOUND!")

        except (TimeoutError, SocketTimeout):

            # ------------------------------------------------
            # IF ORACLE TIMES OUT
            # ------------------------------------------------

            print(
                "!!! Oracle timeout for",
                parameter_name,
                "=",
                value
            )

            print("Skipping this configuration.")

            continue

        except HTTPError as e:

            # ------------------------------------------------
            # HANDLE HTTP ERRORS
            # ------------------------------------------------

            print(
                "!!! HTTP error for",
                parameter_name,
                "=",
                value
            )

            print("Status code:", e.code)

            continue

    # --------------------------------------------------------
    # KEEP BEST VALUE FOUND
    # --------------------------------------------------------

    current_params[parameter_name] = parameter_best_value

    best_params = current_params.copy()
    best_loss = parameter_best_loss

    print("\n" + "-" * 60)
    print("Best", parameter_name, "=", parameter_best_value)
    print("Best loss =", best_loss)
    print("-" * 60)

    return current_params

In [25]:
# ============================================================
# RESTORE CURRENT BEST RESULT
# ============================================================

best_params = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.8,
    "n_estimators": 12,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

best_loss = 6.82180027961698

current_params = best_params.copy()

print("Current best parameters:")
print(best_params)

print("\nCurrent best loss:")
print(best_loss)

print("\nOracle calls attempted so far:")
print(api_call_count)

Current best parameters:
{'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}

Current best loss:
6.82180027961698

Oracle calls attempted so far:
7


In [26]:
# ============================================================
# CONTINUE MAX_DEPTH OPTIMIZATION
# ============================================================

current_params = optimize_parameter(
    current_params,
    "max_depth"
)

print("\n" + "=" * 60)
print("MAX_DEPTH SEARCH COMPLETE")
print("=" * 60)

print("Best parameters:")
print(best_params)

print("\nBest loss:")
print(best_loss)

print("\nOracle calls attempted:")
print(api_call_count)


Optimizing: max_depth
Current value: 7
Current best loss: 6.82180027961698

Trying max_depth = 3
Already tested -> using cached result
max_depth = 3 --> loss = 10.347232

Trying max_depth = 4
Already tested -> using cached result
max_depth = 4 --> loss = 8.04029

Trying max_depth = 5
New Oracle query. Call number: 8
max_depth = 5 --> loss = 7.29016

Trying max_depth = 7
Already tested -> using cached result
max_depth = 7 --> loss = 6.8218

Trying max_depth = 10
New Oracle query. Call number: 9
max_depth = 10 --> loss = 7.397085

Trying max_depth = None
New Oracle query. Call number: 10
max_depth = None --> loss = 7.039238

------------------------------------------------------------
Best max_depth = 7
Best loss = 6.82180027961698
------------------------------------------------------------

MAX_DEPTH SEARCH COMPLETE
Best parameters:
{'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}

Best loss:
6.82180027961698



In [27]:
# ============================================================
# OPTIMIZE min_samples_leaf
# ============================================================

current_params = optimize_parameter(
    current_params,
    "min_samples_leaf"
)

print("\n" + "=" * 60)
print("MIN_SAMPLES_LEAF SEARCH COMPLETE")
print("=" * 60)

print("Best parameters:")
print(best_params)

print("\nBest loss:")
print(best_loss)

print("\nOracle calls attempted:")
print(api_call_count)


Optimizing: min_samples_leaf
Current value: 2
Current best loss: 6.82180027961698

Trying min_samples_leaf = 1
New Oracle query. Call number: 11
min_samples_leaf = 1 --> loss = 6.96719

Trying min_samples_leaf = 2
Already tested -> using cached result
min_samples_leaf = 2 --> loss = 6.8218

Trying min_samples_leaf = 4
New Oracle query. Call number: 12
min_samples_leaf = 4 --> loss = 6.960661

Trying min_samples_leaf = 8
New Oracle query. Call number: 13
min_samples_leaf = 8 --> loss = 7.281385

------------------------------------------------------------
Best min_samples_leaf = 2
Best loss = 6.82180027961698
------------------------------------------------------------

MIN_SAMPLES_LEAF SEARCH COMPLETE
Best parameters:
{'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}

Best loss:
6.82180027961698

Oracle calls attempted:
13


In [28]:
# ============================================================
# IMPORTS FOR SIMULATED ANNEALING
# ============================================================

import random
import math

In [29]:
# ============================================================
# CREATE A NEIGHBOURING CONFIGURATION
# ============================================================

def create_neighbor(params):
    """
    Create a new configuration by changing ONE random
    hyperparameter.

    This allows the optimizer to explore nearby configurations.
    """

    # Make a copy of the current configuration
    neighbor = params.copy()

    # Randomly choose ONE hyperparameter
    parameter_name = random.choice(list(SPACE.keys()))

    # Get all allowed values for that parameter
    allowed_values = SPACE[parameter_name]

    # Current value
    current_value = neighbor[parameter_name]

    # Choose a different value
    possible_values = [
        value
        for value in allowed_values
        if value != current_value
    ]

    # Randomly select the new value
    new_value = random.choice(possible_values)

    # Change the selected parameter
    neighbor[parameter_name] = new_value

    return neighbor

In [30]:
# ============================================================
# TEST NEIGHBOUR GENERATION
# ============================================================

neighbor = create_neighbor(best_params)

print("Current best:")
print(best_params)

print("\nGenerated neighbour:")
print(neighbor)

Current best:
{'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}

Generated neighbour:
{'bootstrap': False, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}


In [31]:
# ============================================================
# SIMULATED ANNEALING OPTIMIZER
# ============================================================

def simulated_annealing(
    initial_params,
    initial_loss,
    max_new_calls=25,
    initial_temperature=1.0,
    cooling_rate=0.90
):
    """
    Manual Simulated Annealing optimizer.

    Parameters
    ----------
    initial_params:
        Configuration from which optimization starts.

    initial_loss:
        Loss of the starting configuration.

    max_new_calls:
        Maximum number of NEW Oracle queries allowed.

    initial_temperature:
        Starting temperature. Higher temperature means
        more exploration.

    cooling_rate:
        Controls how quickly exploration decreases.
    """

    global api_call_count
    global best_params
    global best_loss

    # --------------------------------------------------------
    # CURRENT SOLUTION
    # --------------------------------------------------------

    current_params = initial_params.copy()
    current_loss = initial_loss

    # Global best solution found so far
    global_best_params = initial_params.copy()
    global_best_loss = initial_loss

    # Store the number of calls when we start
    starting_calls = api_call_count

    # Initial temperature
    temperature = initial_temperature

    print("=" * 60)
    print("STARTING SIMULATED ANNEALING")
    print("=" * 60)

    print("Starting loss:", current_loss)
    print("Starting parameters:")
    print(current_params)

    # --------------------------------------------------------
    # OPTIMIZATION LOOP
    # --------------------------------------------------------

    while (api_call_count - starting_calls) < max_new_calls:

        # ----------------------------------------------------
        # CREATE A NEIGHBOUR
        # ----------------------------------------------------

        candidate = create_neighbor(current_params)

        # ----------------------------------------------------
        # EVALUATE CANDIDATE
        # ----------------------------------------------------

        try:
            candidate_loss = evaluate(candidate)

        except (TimeoutError, SocketTimeout):

            print("\nOracle timeout.")
            print("Skipping candidate.")

            continue

        except HTTPError as e:

            print("\nOracle HTTP error:", e.code)
            print("Skipping candidate.")

            continue

        # ----------------------------------------------------
        # CALCULATE LOSS DIFFERENCE
        # ----------------------------------------------------

        difference = candidate_loss - current_loss

        # ----------------------------------------------------
        # ACCEPTANCE DECISION
        # ----------------------------------------------------

        if difference <= 0:

            # Candidate is better
            accept = True

        else:

            # Candidate is worse.
            # It can still be accepted depending on temperature.

            acceptance_probability = math.exp(
                -difference / temperature
            )

            random_number = random.random()

            accept = random_number < acceptance_probability

        # ----------------------------------------------------
        # ACCEPT OR REJECT
        # ----------------------------------------------------

        if accept:

            current_params = candidate.copy()
            current_loss = candidate_loss

            print(
                "\nAccepted candidate | "
                "Loss:",
                round(current_loss, 6),
                "| Temperature:",
                round(temperature, 4)
            )

        else:

            print(
                "\nRejected candidate | "
                "Candidate loss:",
                round(candidate_loss, 6),
                "| Current loss:",
                round(current_loss, 6)
            )

        # ----------------------------------------------------
        # UPDATE GLOBAL BEST
        # ----------------------------------------------------

        if candidate_loss < global_best_loss:

            global_best_loss = candidate_loss
            global_best_params = candidate.copy()

            print(">>> GLOBAL BEST IMPROVED!")
            print(">>> Best loss:", global_best_loss)

        # ----------------------------------------------------
        # COOL DOWN
        # ----------------------------------------------------

        temperature *= cooling_rate

        # Prevent temperature becoming exactly zero
        temperature = max(temperature, 0.001)

        # ----------------------------------------------------
        # DISPLAY PROGRESS
        # ----------------------------------------------------

        print(
            "Oracle calls used:",
            api_call_count - starting_calls
        )

    # --------------------------------------------------------
    # SAVE GLOBAL BEST
    # --------------------------------------------------------

    best_params = global_best_params.copy()
    best_loss = global_best_loss

    print("\n" + "=" * 60)
    print("SIMULATED ANNEALING COMPLETE")
    print("=" * 60)

    print("Best parameters:")
    print(best_params)

    print("\nBest loss:")
    print(best_loss)

    print("\nTotal Oracle calls:")
    print(api_call_count)

    return best_params, best_loss

In [32]:
# ============================================================
# RUN SIMULATED ANNEALING
# ============================================================

best_params, best_loss = simulated_annealing(
    initial_params=best_params,
    initial_loss=best_loss,
    max_new_calls=25,
    initial_temperature=1.0,
    cooling_rate=0.90
)

STARTING SIMULATED ANNEALING
Starting loss: 6.82180027961698
Starting parameters:
{'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}
Already tested -> using cached result

Accepted candidate | Loss: 6.96719 | Temperature: 1.0
Oracle calls used: 0
New Oracle query. Call number: 14

Accepted candidate | Loss: 7.193621 | Temperature: 0.9
Oracle calls used: 1
New Oracle query. Call number: 15

Rejected candidate | Candidate loss: 7.281385 | Current loss: 7.193621
Oracle calls used: 2
Already tested -> using cached result

Rejected candidate | Candidate loss: 7.281385 | Current loss: 7.193621
Oracle calls used: 2
New Oracle query. Call number: 16

Accepted candidate | Loss: 7.484829 | Temperature: 0.6561
Oracle calls used: 3
New Oracle query. Call number: 17

Accepted candidate | Loss: 8.231597 | Temperature: 0.5905
Oracle calls used: 4
New Oracle query. Call number: 18

Accepted candidate | Loss: 7.882777 | Temperatur

In [33]:
# ============================================================
# FINAL OPTIMIZATION RESULT
# ============================================================

print("\n" + "=" * 70)
print("FINAL OPTIMIZATION RESULT")
print("=" * 70)

print("\nBest Optimized Hyperparameters:")
print("-" * 70)

for parameter, value in best_params.items():
    print(f"{parameter:<20} : {value}")

print("-" * 70)

print(f"\nBest Loss              : {best_loss:.10f}")
print(f"Total Oracle Calls    : {api_call_count}")

print("\n" + "=" * 70)
print("OPTIMIZATION COMPLETED")
print("=" * 70)


FINAL OPTIMIZATION RESULT

Best Optimized Hyperparameters:
----------------------------------------------------------------------
bootstrap            : True
max_depth            : 7
max_features         : 0.8
n_estimators         : 12
min_samples_leaf     : 2
min_samples_split    : 5
----------------------------------------------------------------------

Best Loss              : 6.8218002796
Total Oracle Calls    : 38

OPTIMIZATION COMPLETED


In [34]:
# ============================================================
# SECOND STAGE: TARGETED LOCAL EXPLORATION
# ============================================================

import itertools

# Current best configuration
base_params = best_params.copy()

# Number of NEW Oracle calls we are willing to spend
MAX_EXTRA_CALLS = 20

# Store how many calls we have before this stage
calls_before = api_call_count

print("=" * 70)
print("SECOND-STAGE TARGETED SEARCH")
print("=" * 70)

print("\nStarting best loss:", best_loss)
print("Starting parameters:")
print(base_params)

# ------------------------------------------------------------
# Create candidates by changing TWO parameters at a time
# ------------------------------------------------------------

candidate_configs = []

parameter_names = list(SPACE.keys())

for i in range(len(parameter_names)):

    for j in range(i + 1, len(parameter_names)):

        param1 = parameter_names[i]
        param2 = parameter_names[j]

        # Try all allowed values for the two selected parameters
        for value1 in SPACE[param1]:

            for value2 in SPACE[param2]:

                candidate = base_params.copy()

                candidate[param1] = value1
                candidate[param2] = value2

                # Don't include the exact current best
                if candidate != base_params:
                    candidate_configs.append(candidate)


# ------------------------------------------------------------
# Shuffle candidates
# ------------------------------------------------------------

random.shuffle(candidate_configs)


# ------------------------------------------------------------
# Evaluate candidates until API budget is reached
# ------------------------------------------------------------

for candidate in candidate_configs:

    # Stop after the additional API-call budget
    if api_call_count - calls_before >= MAX_EXTRA_CALLS:
        break

    key = tuple(sorted(candidate.items()))

    # --------------------------------------------------------
    # If already tested, use cache and don't spend a call
    # --------------------------------------------------------

    if key in results_cache:

        loss = results_cache[key]

        print("\nCached configuration")
        print("Loss:", round(loss, 6))

    else:

        # New Oracle query
        try:

            loss = evaluate(candidate)

            print("\nNew configuration")
            print("Parameters:", candidate)
            print("Loss:", round(loss, 6))

        except (TimeoutError, SocketTimeout):

            print("\nOracle timeout.")
            print("Skipping configuration.")

            continue

        except HTTPError as e:

            print("\nOracle HTTP error:", e.code)
            print("Skipping configuration.")

            continue

    # --------------------------------------------------------
    # Update global best
    # --------------------------------------------------------

    if loss < best_loss:

        best_loss = loss
        best_params = candidate.copy()

        print("\n>>> NEW GLOBAL BEST <<<")
        print("Best loss:", best_loss)
        print("Best parameters:", best_params)


# ------------------------------------------------------------
# Final result of this stage
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SECOND-STAGE SEARCH COMPLETE")
print("=" * 70)

print("\nBest parameters:")
print(best_params)

print("\nBest loss:")
print(best_loss)

print("\nTotal Oracle calls:")
print(api_call_count)

print("\nAdditional calls used:")
print(api_call_count - calls_before)

SECOND-STAGE TARGETED SEARCH

Starting best loss: 6.82180027961698
Starting parameters:
{'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}
New Oracle query. Call number: 39

New configuration
Parameters: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 10}
Loss: 6.758685

>>> NEW GLOBAL BEST <<<
Best loss: 6.75868505814689
Best parameters: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 10}
New Oracle query. Call number: 40

New configuration
Parameters: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.4, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 6.881463
New Oracle query. Call number: 41

New configuration
Parameters: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 8, 'min_samples_leaf': 4, 'min_samples_split':

In [35]:
# ============================================================
# FINAL BEST RESULT
# ============================================================

print("\n" + "=" * 70)
print("FINAL OPTIMIZATION RESULT")
print("=" * 70)

print("\nBest Optimized Hyperparameters:")
print("-" * 70)

for parameter, value in best_params.items():
    print(f"{parameter:<20} : {value}")

print("-" * 70)

print(f"Best Loss           : {best_loss:.10f}")
print(f"Total Oracle Calls  : {api_call_count}")

print("=" * 70)


FINAL OPTIMIZATION RESULT

Best Optimized Hyperparameters:
----------------------------------------------------------------------
bootstrap            : True
max_depth            : 7
max_features         : 1
n_estimators         : 12
min_samples_leaf     : 2
min_samples_split    : 5
----------------------------------------------------------------------
Best Loss           : 6.6924627800
Total Oracle Calls  : 58
